# Kyrgyz (kir) — Full Neural Pipeline with Cyrillic/Latin Support

Kyrgyz supports the full Stanza neural pipeline via the KTMU UD treebank (POS, lemma, depparse), Apertium FST morphology (Stable quality), and bidirectional Cyrillic↔Latin transliteration. NLLB-200 provides embeddings and translation.

In [ ]:
# Install TurkicNLP
# pip install turkicnlp          # core (tokenization, transliteration)
# pip install "turkicnlp[stanza]"  # adds POS, lemma, depparse, NER
# pip install "turkicnlp[nllb]"    # adds cross-lingual embeddings + translation
# pip install "turkicnlp[all]"     # all optional dependencies

In [ ]:
import turkicnlp
from turkicnlp import Pipeline

## 1. Download Models

In [ ]:
turkicnlp.download('kir')

## 2. Script Detection and Cyrillic ↔ Latin Transliteration

In [ ]:
from turkicnlp.scripts import Script
from turkicnlp.scripts.detector import detect_script
from turkicnlp.scripts.transliterator import Transliterator

print("=" * 70)
print("KYRGYZ COMPREHENSIVE TRANSLITERATION")
print("=" * 70)
print()

cyrl = "Бишкек Кыргызстандын башкаласы."
print(f"Original (Cyrillic): {cyrl}")
print()

# Direction 1: Cyrillic → Turkic Common Alphabet (Latin)
print("1. Cyrillic → Turkic Common Alphabet (Latin):")
try:
    t1 = Transliterator("kir", source=Script.CYRILLIC, target=Script.COMMON_TURKIC)
    common = t1.transliterate(cyrl)
    print(f"   {common}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

# Direction 2: Turkic Common → Cyrillic (reverse)
print("2. Turkic Common (Latin) → Cyrillic:")
try:
    t2 = Transliterator("kir", source=Script.COMMON_TURKIC, target=Script.CYRILLIC)
    back_to_cyrl = t2.transliterate(common if 'common' in locals() else "Bishkek Kyrgyzstandyn bashkalasy.")
    print(f"   {back_to_cyrl}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

# Direction 3: Cyrillic → Latin (explicit)
print("3. Cyrillic → Latin (explicit):")
try:
    t3 = Transliterator("kir", source=Script.CYRILLIC, target=Script.LATIN)
    latin = t3.transliterate(cyrl)
    print(f"   {latin}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

# Direction 4: Latin → Cyrillic
print("4. Latin → Cyrillic:")
try:
    t4 = Transliterator("kir", source=Script.LATIN, target=Script.CYRILLIC)
    back_to_cyrl_explicit = t4.transliterate(latin if 'latin' in locals() else "Bishkek Kyrgyzstandyn bashkalasy.")
    print(f"   {back_to_cyrl_explicit}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

print("=" * 70)
print("Kyrgyz Scripts Supported:")
print("  • Cyrillic (primary, Soviet legacy)")
print("  • Latin (COMMON_TURKIC standard, emerging)")
print("=" * 70)

## 3. Morphological Analysis (Apertium FST)

In [ ]:
nlp_morph = Pipeline(
    "kir",
    processors=["tokenize", "morph"],
    morph_backend="apertium",
    script="Cyrl",
)
doc = nlp_morph("Мен мектепке барам.")
for w in doc.words:
    print(f"{w.text:<18} lemma={w.lemma:<12} feats={w.feats}")

## 4. POS Tagging, Lemmatisation, and Dependency Parsing (KTMU treebank)

In [ ]:
nlp_parse = Pipeline(
    "kir",
    processors=["tokenize", "pos", "lemma", "depparse"],
    script="Cyrl",
)

doc = nlp_parse("Ала-Тоо тоолор Кыргызстанда жайгашкан.")
print(f"{'Word':<20} {'UPOS':<8} {'Lemma':<20} {'Deprel'}")
print("-" * 60)
for w in doc.words:
    print(f"{w.text:<20} {w.upos:<8} {w.lemma:<20} {w.deprel}")

## 5. Full Pipeline with CoNLL-U Export

In [ ]:
nlp_full = Pipeline(
    "kir",
    processors=["tokenize", "morph", "pos", "lemma", "depparse"],
    morph_backend="apertium",
    script="Cyrl",
)
doc = nlp_full("Кыргыз тили Кыргызстандын расмий тили болуп эсептелет.")
print(doc.to_conllu())

## 6. Embeddings and Translation

In [ ]:
turkicnlp.download("kir", processors=["translate"])
trans = Pipeline("kir", processors=["translate"], translate_tgt_lang="eng_Latn")
doc = trans("Кыргызстан — Борбордук Азиядагы мамлекет.")
print("EN:", doc.translation)